# Séance 3 : Analyse et visualisation de réseaux avec Python

**Durée :** 1 journée (7h)
**Public :** Chercheurs et chercheuses en SHS ayant suivi les Séances 1 (bases de Python/pandas) et 2 (fouille de texte)
**Données :** `auc.csv` (parcours d'étudiants — Séance 1) et `olympic_corpus.csv` (corpus de presse — Séance 2)

---

## Déroulé de la journée (indicatif, à ajuster selon le rythme du groupe)

| Horaire | Durée | Bloc | Contenu |
|---|---|---|---|
| 9h00 – 9h15 | 15 min | 0. Introduction | Qu'est-ce qu'un réseau ? Vocabulaire, cas d'usage en SHS |
| 9h15 – 10h00 | 45 min | 1. Prise en main de NetworkX | Créer, manipuler et dessiner un premier graphe |
| 10h00 – 11h15 | 75 min | 2. Réseau d'affiliation (`auc.csv`) | D'un tableau à un réseau bipartite |
| 11h15 – 11h30 | 15 min | ☕ Pause | |
| 11h30 – 12h30 | 60 min | 3. Mesures de réseau | Densité, centralités, projection |
| 12h30 – 13h30 | 60 min | 🍽️ Déjeuner | |
| 13h30 – 14h45 | 75 min | 4. Réseau de co-occurrence (`olympic_corpus.csv`) | D'un corpus de texte à un réseau d'entités |
| 14h45 – 15h00 | 15 min | ☕ Pause | |
| 15h00 – 16h00 | 60 min | 5. Communautés & visualisation avancée | Louvain, mise en forme, export interactif |
| 16h00 – 16h40 | 40 min | 6. Mini-projet de synthèse | Exercice récapitulatif en autonomie |
| 16h40 – 17h00 | 20 min | Aide-mémoire & clôture | Ressources, questions |

> 💡 **Comment utiliser ce notebook** : chaque section alterne explications, démonstrations et exercices **🧪 À vous de jouer**. Faites les exercices avant de regarder la solution, cachée dans un bloc repliable juste en dessous.

> ⚠️ **Installations nécessaires** :
> ```bash
> pip install pandas matplotlib seaborn networkx pyvis spacy
> python -m spacy download en_core_web_sm
> ```
> `networkx` est la bibliothèque de référence pour l'analyse de réseaux en Python (l'équivalent direct de `igraph` ou `tidygraph` en R). `pyvis` sert à produire des visualisations **interactives** exportables en HTML.

> 📁 **Données** : placez `auc.csv` et `olympic_corpus.csv` dans un dossier `data/` à côté de ce notebook (les mêmes fichiers que lors des Séances 1 et 2).


# 0. Introduction : qu'est-ce qu'un réseau ?

Un **réseau** (ou **graphe**) est une structure de données composée de :

- des **nœuds** (*nodes*, ou *vertices*) : les entités étudiées (personnes, institutions, lieux, mots...) ;
- des **liens** (*edges*, ou *arêtes*) : les relations entre ces entités (collaboration, affiliation, co-occurrence, correspondance...).

C'est une structure fondamentalement différente du tableau (dataframe) que nous avons manipulé lors des deux premières séances : plutôt que des lignes indépendantes, on s'intéresse ici aux **relations** entre unités. C'est un changement de perspective précieux pour les SHS : il permet d'étudier des phénomènes relationnels — réseaux de sociabilité, circulations, filiations institutionnelles, structures argumentatives — qui échappent à l'analyse tabulaire classique.

## Vocabulaire de base

| Terme | Description |
|---|---|
| **Nœud** (*node*/*vertex*) | Une entité du réseau (un individu, une institution, un lieu...) |
| **Lien** (*edge*) | Une relation entre deux nœuds |
| **Graphe non-orienté** | Les liens n'ont pas de direction (ex. « A et B ont co-écrit un article ») |
| **Graphe orienté** (*directed*) | Les liens ont un sens (ex. « A cite B ») |
| **Graphe pondéré** (*weighted*) | Chaque lien porte un poids (ex. nombre de collaborations) |
| **Réseau biparti** (*bipartite*) | Deux types de nœuds distincts, les liens n'existant qu'*entre* les deux types (ex. individus ↔ institutions), jamais entre nœuds d'un même type |
| **Projection** | Transformation d'un réseau biparti en réseau à un seul type de nœuds (ex. institutions liées entre elles via les individus qu'elles partagent) |
| **Degré** (*degree*) | Nombre de liens d'un nœud |
| **Centralité** | Famille de mesures quantifiant l'importance d'un nœud dans le réseau (voir section 3) |
| **Communauté** | Sous-groupe de nœuds densément connectés entre eux, plus faiblement au reste du réseau |

## Pourquoi l'analyse de réseau en SHS ?

L'analyse de réseau (*Social Network Analysis*, SNA) a une longue tradition en sciences sociales et en histoire — des travaux fondateurs de Padgett et Ansell sur les réseaux de mariage et d'affaires des Médicis, aux études contemporaines de circulation des idées, des correspondances savantes, ou des filières migratoires. Aujourd'hui, nous allons appliquer ces méthodes à deux corpus déjà connus :

1. **`auc.csv`** (Séance 1) : nous allons transformer les parcours individuels des étudiants en un **réseau d'affiliation** entre universités et employeurs.
2. **`olympic_corpus.csv`** (Séance 2) : nous allons transformer les entités nommées extraites en Séance 2 en un **réseau de co-occurrence** — quels acteurs, lieux et institutions apparaissent ensemble dans les mêmes articles ?

## La bibliothèque NetworkX

**NetworkX** est la bibliothèque de référence pour la création, la manipulation et l'analyse de réseaux en Python.


In [ ]:
import pandas as pd
import numpy as np
import re
import itertools
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("NetworkX version :", nx.__version__)
print("Bibliothèques chargées avec succès ✅")


# 1. Prise en main de NetworkX

Avant de construire des réseaux à partir de nos données réelles, familiarisons-nous avec les objets de base de NetworkX sur un petit exemple créé à la main.

## Créer un graphe

Il existe quatre classes de graphes principales dans NetworkX :

| Classe | Orienté ? | Liens multiples autorisés ? |
|---|---|---|
| `nx.Graph()` | Non | Non |
| `nx.DiGraph()` | Oui | Non |
| `nx.MultiGraph()` | Non | Oui |
| `nx.MultiDiGraph()` | Oui | Oui |

Pour la majorité de nos usages aujourd'hui, un simple `nx.Graph()` (non-orienté) suffira.


In [ ]:
G = nx.Graph()

# Ajouter des nœuds
G.add_node("Alice")
G.add_nodes_from(["Bob", "Chen", "Deepa", "Emeka"])

# Ajouter des liens (les nœuds sont créés automatiquement s'ils n'existent pas encore)
G.add_edge("Alice", "Bob")
G.add_edges_from([
    ("Alice", "Chen"),
    ("Bob", "Chen"),
    ("Chen", "Deepa"),
    ("Deepa", "Emeka"),
])

print("Nœuds :", G.nodes())
print("Liens :", G.edges())
print("Nombre de nœuds :", G.number_of_nodes())
print("Nombre de liens :", G.number_of_edges())


## Dessiner un graphe

La fonction `nx.draw()` fournit une visualisation rapide (mais assez basique) directement avec matplotlib :

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

pos = nx.spring_layout(G, seed=42)   # positionnement automatique des nœuds
nx.draw(
    G, pos, ax=ax,
    with_labels=True, node_color="#4C72B0", font_color="white",
    node_size=800, edge_color="gray", width=1.5
)
ax.set_title("Mon premier réseau")
plt.show()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- <code>nx.spring_layout()</code> calcule automatiquement une disposition des nœuds fondée sur un algorithme physique (les nœuds se repoussent, les liens agissent comme des ressorts) — c'est l'algorithme de disposition le plus utilisé. On fixe <code>seed=42</code> pour obtenir un résultat reproductible d'une exécution à l'autre.
- D'autres dispositions existent : <code>nx.circular_layout()</code>, <code>nx.kamada_kawai_layout()</code>, <code>nx.shell_layout()</code>...
</div>

## Attributs des nœuds et des liens

On peut attacher des **attributs** aux nœuds (par exemple, une catégorie) et aux liens (par exemple, un poids) :

In [ ]:
G.nodes["Alice"]["role"] = "Historian"
G.nodes["Bob"]["role"] = "Sociologist"
G.nodes["Chen"]["role"] = "Historian"
G.nodes["Deepa"]["role"] = "Linguist"
G.nodes["Emeka"]["role"] = "Sociologist"

G["Alice"]["Bob"]["weight"] = 3     # 3 collaborations
G["Alice"]["Chen"]["weight"] = 1
G["Bob"]["Chen"]["weight"] = 5
G["Chen"]["Deepa"]["weight"] = 2
G["Deepa"]["Emeka"]["weight"] = 1

# On peut relire ces attributs
print(G.nodes(data=True))
print(G.edges(data=True))


On peut utiliser ces attributs pour enrichir la visualisation — par exemple, colorer les nœuds selon leur rôle et faire varier l'épaisseur des liens selon leur poids :

In [ ]:
role_colors = {"Historian": "#4C72B0", "Sociologist": "#DD8452", "Linguist": "#55A868"}
node_colors = [role_colors[G.nodes[n]["role"]] for n in G.nodes()]
edge_widths = [G[u][v]["weight"] for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(6, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G, pos, ax=ax, with_labels=True,
    node_color=node_colors, font_color="white",
    node_size=800, edge_color="gray", width=edge_widths
)
ax.set_title("Réseau coloré par discipline, pondéré par le nombre de collaborations")
plt.show()


## Le degré d'un nœud

Le **degré** d'un nœud est le nombre de liens qui lui sont rattachés — la mesure la plus simple, mais souvent très informative :

In [ ]:
dict(G.degree())


## 🧪 À vous de jouer — Exercice 1

1. Créez un nouveau graphe non-orienté `G2` avec au moins 6 nœuds et 7 liens de votre choix (par exemple, un petit réseau de personnages d'un roman, ou de collègues d'un même laboratoire).
2. Attribuez à chaque nœud un attribut `group` (au moins deux groupes différents).
3. Dessinez ce graphe en colorant les nœuds selon leur groupe.
4. Quel est le nœud avec le degré le plus élevé ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
G2 = nx.Graph()
G2.add_edges_from([
    ("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"),
    ("D", "E"), ("D", "F"), ("E", "F")
])

groups = {"A": "group1", "B": "group1", "C": "group1", "D": "group2", "E": "group2", "F": "group2"}
nx.set_node_attributes(G2, groups, "group")

colors = {"group1": "#4C72B0", "group2": "#DD8452"}
node_colors = [colors[G2.nodes[n]["group"]] for n in G2.nodes()]

fig, ax = plt.subplots(figsize=(6, 6))
pos = nx.spring_layout(G2, seed=1)
nx.draw(G2, pos, ax=ax, with_labels=True, node_color=node_colors, font_color="white", node_size=700)
plt.show()

max(dict(G2.degree()).items(), key=lambda x: x[1])   # nœud de plus haut degré
```
</details>


# 2. Construire un réseau d'affiliation à partir de `auc.csv`

Nous retrouvons le jeu de données de la Séance 1 : les parcours d'étudiants formés aux États-Unis, avec leur université de formation et leur employeur après diplomation. Aujourd'hui, nous allons le traiter comme un véritable **réseau**, ce qui nous permettra d'aller plus loin : mesurer les liens entre individus et institutions, détecter des regroupements, etc.

## Recharger et nettoyer les données

On reprend les étapes de nettoyage vues en Séance 1 :

In [ ]:
auc = pd.read_csv("data/auc.csv", sep=";", encoding="utf-8")

auc_simple = auc[["Name_full", "University", "Field_main", "Employer_main", "Sector_1"]].rename(columns={
    "Name_full": "Name",
    "Field_main": "Field",
    "Employer_main": "Employer",
    "Sector_1": "Sector"
})

auc_simple.head()


## Un réseau biparti : universités ↔ employeurs

Un **réseau biparti** (*bipartite network*) comporte deux types de nœuds distincts, les liens n'existant qu'*entre* les deux types. Ici : d'un côté les **universités**, de l'autre les **employeurs**, et un lien entre une université et un employeur chaque fois qu'un étudiant est passé de l'une à l'autre.

### Étape 1 : construire la table des liens (edge list)

On part des couples uniques (étudiant, université, employeur), puis on compte, pour chaque couple (université, employeur), le nombre d'étudiants qui l'ont emprunté :

In [ ]:
links = auc_simple[["Name", "University", "Employer"]].drop_duplicates()
links = links.dropna(subset=["University", "Employer"])

edges = links.groupby(["University", "Employer"]).size().reset_index(name="weight")
edges = edges.sort_values("weight", ascending=False)

edges.head(10)


In [ ]:
B = nx.Graph()

universities = edges["University"].unique()
employers = edges["Employer"].unique()

B.add_nodes_from(universities, bipartite=0, node_type="University")
B.add_nodes_from(employers, bipartite=1, node_type="Employer")

for _, row in edges.iterrows():
    B.add_edge(row["University"], row["Employer"], weight=row["weight"])

print(f"{B.number_of_nodes()} nœuds ({len(universities)} universités + {len(employers)} employeurs)")
print(f"{B.number_of_edges()} liens")


### Étape 3 : visualiser le réseau biparti

Le réseau complet est trop dense pour être lisible d'un seul coup d'œil. On commence donc par filtrer les liens les plus significatifs (poids ≥ 2, c'est-à-dire au moins deux étudiants ayant emprunté ce chemin) :

In [ ]:
strong_edges = edges[edges["weight"] >= 2]

B_strong = nx.Graph()
strong_univ = strong_edges["University"].unique()
strong_empl = strong_edges["Employer"].unique()

B_strong.add_nodes_from(strong_univ, bipartite=0, node_type="University")
B_strong.add_nodes_from(strong_empl, bipartite=1, node_type="Employer")
for _, row in strong_edges.iterrows():
    B_strong.add_edge(row["University"], row["Employer"], weight=row["weight"])

print(f"{B_strong.number_of_nodes()} nœuds, {B_strong.number_of_edges()} liens (seuil >= 2)")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

pos = nx.spring_layout(B_strong, seed=42, k=0.5)

node_colors = ["#4C72B0" if B_strong.nodes[n]["node_type"] == "University" else "#DD8452" for n in B_strong.nodes()]
edge_widths = [B_strong[u][v]["weight"] * 0.8 for u, v in B_strong.edges()]

nx.draw_networkx_nodes(B_strong, pos, node_color=node_colors, node_size=250, ax=ax)
nx.draw_networkx_edges(B_strong, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(B_strong, pos, font_size=6, ax=ax)

ax.set_title("Bipartite network: Universities (blue) and Employers (orange)")
ax.axis("off")

plt.figtext(
    0.5, 0.01,
    "Note : seuls les liens de poids ≥ 2 sont représentés.",
    ha="center",
    fontsize=9
)

plt.show()
plt.show()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Le filtrage est une étape normale et nécessaire</b> en visualisation de réseau : un réseau complet issu de données réelles est presque toujours trop dense pour être lu visuellement. On filtre en général selon un seuil de poids, ou en ne conservant que les *n* nœuds les plus connectés — à condition de toujours mentionner ce filtrage dans la légende ou le commentaire, pour ne pas donner une image trompeuse du réseau complet.
</div>

## 🧪 À vous de jouer — Exercice 2

1. Combien y a-t-il, au total, de couples (université, employeur) uniques dans `edges` (avant filtrage) ?
2. Quels sont les 5 employeurs ayant recruté des diplômés du plus grand nombre d'universités différentes ? (indice : c'est une question sur le **degré** des nœuds employeurs dans `B`)
3. Reconstruisez le réseau biparti en ne conservant que les étudiants du champ `Field == "Engineering"`. Combien de nœuds et de liens ce sous-réseau contient-il ?
4. Redessinez le réseau filtré (`B_strong`) en utilisant un seuil de poids différent (par exemple ≥ 3) : le réseau est-il plus ou moins lisible ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
len(edges)

# Question 2
degrees = dict(B.degree())
employer_degrees = {n: d for n, d in degrees.items() if B.nodes[n]["node_type"] == "Employer"}
sorted(employer_degrees.items(), key=lambda x: -x[1])[:5]

# Question 3
links_eng = auc_simple[auc_simple["Field"] == "Engineering"][["Name", "University", "Employer"]].drop_duplicates().dropna()
edges_eng = links_eng.groupby(["University", "Employer"]).size().reset_index(name="weight")

B_eng = nx.Graph()
B_eng.add_nodes_from(edges_eng["University"].unique(), bipartite=0, node_type="University")
B_eng.add_nodes_from(edges_eng["Employer"].unique(), bipartite=1, node_type="Employer")
for _, row in edges_eng.iterrows():
    B_eng.add_edge(row["University"], row["Employer"], weight=row["weight"])

print(B_eng.number_of_nodes(), B_eng.number_of_edges())

# Question 4
strong_edges_3 = edges[edges["weight"] >= 3]
# ... reconstruire le graphe et le dessiner comme ci-dessus, en remplaçant le seuil
```
</details>


# 3. Mesures de réseau : densité, centralité, projection

Au-delà de la seule visualisation, l'analyse de réseau offre des **mesures quantitatives** pour caractériser la structure d'un graphe et l'importance relative de ses nœuds.

## Densité

La **densité** d'un réseau est la proportion de liens existants par rapport à tous les liens possibles (entre 0 = aucun lien, et 1 = réseau complet où tous les nœuds sont connectés entre eux) :

In [ ]:
nx.density(B)


## Composantes connexes

Une **composante connexe** est un sous-ensemble de nœuds tous reliés entre eux (directement ou indirectement), mais déconnectés du reste du réseau. Un réseau réel comporte souvent plusieurs composantes (des "îlots" isolés) :

In [ ]:
print("Nombre de composantes connexes :", nx.number_connected_components(B))

# Taille de chaque composante
sizes = [len(c) for c in nx.connected_components(B)]
sorted(sizes, reverse=True)[:10]


In [ ]:
# On isole la plus grande composante, souvent la plus intéressante à analyser
largest_cc = max(nx.connected_components(B), key=len)
B_main = B.subgraph(largest_cc).copy()

print(f"Composante principale : {B_main.number_of_nodes()} nœuds, {B_main.number_of_edges()} liens")


## Projection : passer d'un réseau biparti à un réseau à un seul type de nœuds

Le réseau biparti université ↔ employeur est utile, mais il ne permet pas directement de répondre à des questions comme « quelles universités partagent le plus de destinées professionnelles communes ? ». Pour cela, on **projette** le réseau biparti sur l'un des deux côtés — ici, le côté « université » : deux universités sont reliées si elles partagent au moins un employeur, le poids du lien reflétant l'intensité de ce partage.

In [ ]:
from networkx.algorithms import bipartite

univ_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "University"}

G_univ = bipartite.weighted_projected_graph(B, univ_nodes)

print(f"{G_univ.number_of_nodes()} universités, {G_univ.number_of_edges()} liens (employeurs partagés)")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <code>bipartite.weighted_projected_graph(B, univ_nodes)</code> calcule automatiquement, pour chaque paire d'universités, le nombre d'employeurs qu'elles ont en commun, et l'utilise comme poids du nouveau lien. C'est l'équivalent réseau de la « déduplication + comptage » que l'on ferait à la main avec un `.groupby()`.
</div>

## Les mesures de centralité

La **centralité** est une famille de mesures visant à quantifier l'importance d'un nœud, selon différentes définitions de ce qu'« être important » signifie dans un réseau :

| Mesure | Ce qu'elle capture | Question posée |
|---|---|---|
| **Degré** (*degree centrality*) | Nombre de connexions directes | « Combien de partenaires ce nœud a-t-il ? » |
| **Intermédiarité** (*betweenness centrality*) | Fréquence à laquelle un nœud se trouve sur le plus court chemin entre deux autres | « Ce nœud sert-il de pont/passage obligé entre d'autres parties du réseau ? » |
| **Proximité** (*closeness centrality*) | Distance moyenne aux autres nœuds | « Ce nœud peut-il atteindre rapidement tout le reste du réseau ? » |
| **Vecteur propre** (*eigenvector centrality*) | Importance pondérée par l'importance de ses voisins | « Ce nœud est-il connecté à d'autres nœuds eux-mêmes importants ? » |

Calculons ces quatre mesures sur le réseau des universités :

In [ ]:
centrality = pd.DataFrame({
    "degree": nx.degree_centrality(G_univ),
    "betweenness": nx.betweenness_centrality(G_univ, weight="weight"),
    "closeness": nx.closeness_centrality(G_univ),
    "eigenvector": nx.eigenvector_centrality(G_univ, weight="weight", max_iter=1000)
})

centrality.sort_values("degree", ascending=False).head(10)


On peut visualiser le réseau des universités en faisant varier la **taille des nœuds** selon leur centralité de degré, pour repérer visuellement les institutions les plus centrales :

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

pos = nx.spring_layout(G_univ, seed=42, k=0.6)
node_sizes = [centrality.loc[n, "degree"] * 4000 + 100 for n in G_univ.nodes()]
edge_widths = [G_univ[u][v]["weight"] * 0.5 for u, v in G_univ.edges()]

nx.draw_networkx_nodes(G_univ, pos, node_size=node_sizes, node_color="#4C72B0", alpha=0.85, ax=ax)
nx.draw_networkx_edges(G_univ, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_univ, pos, font_size=7, ax=ax)

ax.set_title("University network (node size = degree centrality)")
ax.axis("off")
plt.show()


<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Attention à l'interprétation</b> : un nœud avec un fort <b>degré</b> mais une faible <b>intermédiarité</b> est bien connecté localement, sans forcément jouer de rôle de « pont » entre différentes parties du réseau. Inversement, un nœud à faible degré mais forte intermédiarité peut être un intermédiaire stratégique, même peu visible à première vue. Croiser plusieurs mesures de centralité, plutôt que se fier à une seule, donne une image plus riche et plus fiable de la structure du réseau.
</div>

## 🧪 À vous de jouer — Exercice 3

1. Quelle université a la plus forte **intermédiarité** (*betweenness*) ? Est-ce la même que celle qui a le plus fort degré ?
2. Réalisez la même projection, mais du côté des **employeurs** cette fois (`bipartite.weighted_projected_graph(B, employer_nodes)`), et calculez leur centralité de degré.
3. Combien de composantes connexes compte `G_univ` ? Si plus d'une, quelle interprétation en tirer ?
4. **Bonus** : filtrez `G_univ` pour ne garder que les liens de poids ≥ 2 avant de calculer les centralités : les classements changent-ils ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
centrality.sort_values("betweenness", ascending=False).head(5)
centrality.sort_values("degree", ascending=False).head(5)

# Question 2
employer_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "Employer"}
G_empl = bipartite.weighted_projected_graph(B, employer_nodes)
empl_centrality = pd.Series(nx.degree_centrality(G_empl)).sort_values(ascending=False)
empl_centrality.head(10)

# Question 3
nx.number_connected_components(G_univ)

# Question 4 (bonus)
G_univ_strong = nx.Graph(((u, v, d) for u, v, d in G_univ.edges(data=True) if d["weight"] >= 2))
pd.Series(nx.degree_centrality(G_univ_strong)).sort_values(ascending=False).head(10)
```
</details>


# 4. Construire un réseau de co-occurrence à partir du corpus textuel

Changeons de registre : à partir du corpus de presse sur les Jeux Olympiques (Séance 2), nous allons construire un **réseau de co-occurrence** entre entités nommées. L'idée : deux entités (deux personnes, deux lieux, deux organisations) sont reliées si elles apparaissent **dans le même article**. Le poids du lien reflète le nombre d'articles où elles co-apparaissent.

C'est une méthode très utilisée en humanités numériques pour reconstituer, à partir d'un corpus textuel, des réseaux d'acteurs ou de lieux qui ne sont pas explicitement encodés dans les métadonnées.

## Recharger et préparer le corpus

On reprend les étapes de nettoyage de la Séance 2 :

In [ ]:
corpus = pd.read_csv("data/olympic_corpus.csv")
corpus = corpus.drop(columns=["Unnamed: 0"])
corpus["Date"] = pd.to_datetime(corpus["Date"], format="%Y%m%d", errors="coerce")
corpus["Year"] = corpus["Date"].dt.year

def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", text).strip()

corpus["Text_clean"] = corpus["Text"].apply(clean_text)

editorial_types = ["Feature/Article", "General News", "Editorial/Opinion", "Review"]
articles = corpus[corpus["category_clean"].isin(editorial_types)].copy()
articles["n_words"] = articles["Text_clean"].str.split().str.len()

# Comme en Séance 2, on échantillonne pour la démonstration en direct
sample = articles[articles["n_words"] >= 50].sample(n=500, random_state=42).reset_index(drop=True)

len(sample)


## Étape 1 : extraire les entités nommées de chaque article

On réutilise spaCy (Séance 2) pour extraire, pour chaque article, l'ensemble des entités de type `PERSON`, `GPE` (lieu) et `ORG` (organisation) :

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer"])

doc_entities = {}   # DocId -> ensemble d'entités mentionnées dans ce document

for doc_id, doc in zip(sample["DocId"], nlp.pipe(sample["Text_clean"], batch_size=50)):
    ents = set()
    for ent in doc.ents:
        if ent.label_ in ("PERSON", "GPE", "ORG") and len(ent.text) > 2:
            ents.add(ent.text.strip())
    doc_entities[doc_id] = ents

# Exemple pour le premier document
list(doc_entities.values())[0]


## Étape 2 : construire les liens de co-occurrence

Pour chaque article, on relie **toutes les paires possibles** d'entités qui y apparaissent ensemble (`itertools.combinations`), et l'on cumule le poids de chaque paire sur l'ensemble du corpus :

In [ ]:
node_counter = Counter()   # nombre d'articles mentionnant chaque entité
edge_counter = Counter()   # nombre d'articles où chaque paire d'entités co-apparaît

for ents in doc_entities.values():
    for e in ents:
        node_counter[e] += 1
    for a, b in itertools.combinations(sorted(ents), 2):
        edge_counter[(a, b)] += 1

print(f"{len(node_counter)} entités distinctes, {len(edge_counter)} paires de co-occurrence")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- <code>itertools.combinations(sorted(ents), 2)</code> génère toutes les paires possibles d'entités présentes dans un même document (sans répétition, ni paire d'un nœud avec lui-même). Trier les entités avant de générer les paires assure que la paire (A, B) est toujours produite dans le même ordre que (A, B), et jamais aussi comme (B, A) — ce qui éviterait de compter deux fois le même lien.
- <code>Counter</code> (du module <code>collections</code>) est une structure très efficace pour compter des occurrences — un dictionnaire spécialisé qui retourne 0 par défaut pour une clé absente.
</div>

## Étape 3 : filtrer et construire le graphe

Avec plusieurs milliers d'entités distinctes, le réseau complet est totalement illisible (et long à calculer). On se concentre sur les entités les plus fréquentes, et sur les liens les plus robustes :

In [ ]:
TOP_N = 60       # nombre d'entités les plus fréquentes à conserver
MIN_WEIGHT = 2    # nombre minimal d'articles partagés pour garder un lien

top_entities = {e for e, _ in node_counter.most_common(TOP_N)}

G_ent = nx.Graph()
for e in top_entities:
    G_ent.add_node(e, weight=node_counter[e])

for (a, b), w in edge_counter.items():
    if a in top_entities and b in top_entities and w >= MIN_WEIGHT:
        G_ent.add_edge(a, b, weight=w)

print(f"{G_ent.number_of_nodes()} nœuds, {G_ent.number_of_edges()} liens")


## Étape 4 : visualiser le réseau (statique)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

pos = nx.spring_layout(G_ent, seed=42, k=0.5)
node_sizes = [G_ent.nodes[n]["weight"] * 15 for n in G_ent.nodes()]
edge_widths = [G_ent[u][v]["weight"] * 0.3 for u, v in G_ent.edges()]

nx.draw_networkx_nodes(G_ent, pos, node_size=node_sizes, node_color="#C44E52", alpha=0.8, ax=ax)
nx.draw_networkx_edges(G_ent, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_ent, pos, font_size=7, ax=ax)

ax.set_title(f"Entity co-occurrence network (top {TOP_N} entities, min. {MIN_WEIGHT} shared articles)")
ax.axis("off")
plt.show()


## Une première visualisation interactive avec `pyvis`

**pyvis** permet de produire des réseaux **interactifs** (zoom, déplacement des nœuds à la souris, info-bulles) exportés en HTML — bien plus confortables à explorer qu'une image statique pour un réseau de cette taille :

In [ ]:
from pyvis.network import Network

net = Network(height="700px", width="100%", notebook=True, cdn_resources="in_line", bgcolor="white")
net.from_nx(G_ent)

# Active la simulation physique interactive (on peut glisser les nœuds à la souris)
net.show_buttons(filter_=["physics"])

net.show("entity_network.html")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Le fichier <code>entity_network.html</code> peut être ouvert dans n'importe quel navigateur et partagé indépendamment de Python — un bon format pour intégrer une visualisation de réseau interactive dans un carnet de recherche ou un site web.
</div>

## 🧪 À vous de jouer — Exercice 4

1. Quelles sont les 10 entités les plus fréquemment mentionnées (`node_counter.most_common(10)`) ?
2. Recommencez la construction du réseau avec `TOP_N = 40` et `MIN_WEIGHT = 3` : le réseau est-il plus lisible ? Combien de nœuds/liens obtenez-vous ?
3. Quelle paire d'entités co-apparaît dans le plus grand nombre d'articles (indice : `edge_counter.most_common(10)`) ?
4. **Bonus** : reconstruisez le réseau de co-occurrence en vous limitant aux articles publiés entre 1935 et 1936 (contexte des Jeux de Berlin). Le réseau obtenu met-il en avant des entités différentes ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
node_counter.most_common(10)

# Question 2
top_entities_40 = {e for e, _ in node_counter.most_common(40)}
G_ent_2 = nx.Graph()
for e in top_entities_40:
    G_ent_2.add_node(e, weight=node_counter[e])
for (a, b), w in edge_counter.items():
    if a in top_entities_40 and b in top_entities_40 and w >= 3:
        G_ent_2.add_edge(a, b, weight=w)
print(G_ent_2.number_of_nodes(), G_ent_2.number_of_edges())

# Question 3
edge_counter.most_common(10)

# Question 4 (bonus)
sample_berlin = articles[(articles["Year"] >= 1935) & (articles["Year"] <= 1936) & (articles["n_words"] >= 50)]
doc_entities_berlin = {}
for doc_id, doc in zip(sample_berlin["DocId"], nlp.pipe(sample_berlin["Text_clean"], batch_size=50)):
    ents = {ent.text.strip() for ent in doc.ents if ent.label_ in ("PERSON", "GPE", "ORG") and len(ent.text) > 2}
    doc_entities_berlin[doc_id] = ents
# ... puis reconstruire node_counter / edge_counter / G_ent comme ci-dessus
```
</details>


# 5. Détection de communautés et visualisation avancée

## Qu'est-ce qu'une communauté ?

Une **communauté** (ou *cluster*) est un sous-ensemble de nœuds **densément connectés entre eux**, mais plus faiblement reliés au reste du réseau. Détecter des communautés permet de faire émerger des regroupements qui ne sont pas nécessairement visibles dans les métadonnées d'origine — par exemple, des groupes d'entités associées à un même sous-thème.

## L'algorithme de Louvain

L'algorithme de **Louvain** est la méthode de référence pour la détection de communautés : il cherche à maximiser la **modularité** du réseau — une mesure de la qualité d'un découpage en communautés (des liens denses à l'intérieur des groupes, rares entre eux). NetworkX l'intègre nativement :

In [ ]:
communities = nx.community.louvain_communities(G_ent, weight="weight", seed=42)

print(f"{len(communities)} communautés détectées")
for i, com in enumerate(communities):
    print(f"Communauté {i} ({len(com)} nœuds) :", list(com)[:8])


On peut mesurer la **modularité** du découpage obtenu (plus elle est élevée, plus la structure en communautés est marquée) :

In [ ]:
nx.community.modularity(G_ent, communities, weight="weight")


## Colorer le réseau par communauté

Pour visualiser ce découpage, on attribue à chaque nœud la couleur de sa communauté :

In [ ]:
# On crée un dictionnaire nœud -> numéro de communauté
node_to_community = {}
for i, com in enumerate(communities):
    for node in com:
        node_to_community[node] = i

nx.set_node_attributes(G_ent, node_to_community, "community")


In [ ]:
palette = sns.color_palette("tab10", len(communities)).as_hex()

fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(G_ent, seed=42, k=0.5)

node_colors = [palette[G_ent.nodes[n]["community"]] for n in G_ent.nodes()]
node_sizes = [G_ent.nodes[n]["weight"] * 15 for n in G_ent.nodes()]
edge_widths = [G_ent[u][v]["weight"] * 0.3 for u, v in G_ent.edges()]

nx.draw_networkx_nodes(G_ent, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax)
nx.draw_networkx_edges(G_ent, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_ent, pos, font_size=7, ax=ax)

ax.set_title("Entity network colored by detected community")
ax.axis("off")
plt.show()


<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ Comme pour le topic modeling (Séance 2), les communautés détectées sont des regroupements <b>statistiques</b>. Elles doivent être <b>interprétées</b> en revenant aux nœuds qui les composent (et, si besoin, aux articles d'origine via le KWIC de la Séance 2) plutôt qu'être considérées comme des catégories a priori.
</div>

## Visualisation interactive colorée par communauté (pyvis)

On peut intégrer ce même code couleur dans une visualisation interactive pyvis, en ajoutant directement l'attribut `group` à chaque nœud (pyvis colore automatiquement les nœuds selon cet attribut) :

In [ ]:
net2 = Network(height="700px", width="100%", notebook=True, cdn_resources="in_line", bgcolor="white")

for node, data in G_ent.nodes(data=True):
    net2.add_node(node, label=node, group=data["community"], value=data["weight"])

for u, v, data in G_ent.edges(data=True):
    net2.add_edge(u, v, value=data["weight"])

net2.show_buttons(filter_=["physics"])
net2.show("entity_network_communities.html")


## Appliquer la détection de communautés au réseau des universités

La même méthode s'applique bien sûr au réseau `G_univ` de la section 3 : quelles universités forment des groupes aux destinées professionnelles similaires ?

In [ ]:
communities_univ = nx.community.louvain_communities(G_univ, weight="weight", seed=42)

print(f"{len(communities_univ)} communautés d'universités détectées")
for i, com in enumerate(communities_univ):
    print(f"Communauté {i} :", list(com))


## 🧪 À vous de jouer — Exercice 5

1. Combien de nœuds contient la plus grande communauté détectée dans `G_ent` ? Quelles entités en font partie ?
2. Choisissez une communauté et formulez une hypothèse sur ce qui la relie (un lieu ? un événement ? un type d'acteur ?). Vérifiez votre hypothèse en consultant 2-3 articles où ces entités co-apparaissent (indice : reprenez le code de la section 2 de la Séance 2 pour un KWIC, ou filtrez directement `sample` sur les DocId concernés).
3. Recalculez la modularité de `G_univ` avec `communities_univ` : le découpage en communautés est-il marqué (modularité élevée) ou plus flou ?
4. **Bonus** : essayez un autre algorithme de détection de communautés disponible dans NetworkX, par exemple `nx.community.greedy_modularity_communities(G_ent, weight="weight")`. Obtenez-vous un découpage similaire à celui de Louvain ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
sizes = [len(c) for c in communities]
biggest = communities[sizes.index(max(sizes))]
print(len(biggest), biggest)

# Question 3
nx.community.modularity(G_univ, communities_univ, weight="weight")

# Question 4 (bonus)
communities_greedy = nx.community.greedy_modularity_communities(G_ent, weight="weight")
len(communities_greedy)
```
</details>


# 6. Mini-projet de synthèse

Combinons l'ensemble des méthodes vues aujourd'hui sur une **petite investigation en autonomie** (30-40 min). Travaillez seul·e ou en binôme.

## 🧪 Consignes

Choisissez **l'un des deux terrains** (ou les deux, si le temps le permet) :

- **A. Le réseau université–employeur (`auc.csv`)** : concentrez-vous sur un sous-ensemble pertinent (par exemple, un champ disciplinaire, une période, ou une nationalité) et étudiez sa structure relationnelle.
- **B. Le réseau d'entités olympiques (`olympic_corpus.csv`)** : concentrez-vous sur une période, un mot-clé, ou un type d'entité (par exemple, uniquement les `GPE`, pour un réseau de lieux) et étudiez sa structure relationnelle.

Pour le terrain choisi, produisez une mini-analyse en 5 étapes :

1. **Construction** : construisez le réseau pertinent (biparti et/ou projeté et/ou de co-occurrence) à partir d'un sous-ensemble de données que vous justifierez.
2. **Description** : nombre de nœuds, de liens, densité, nombre de composantes connexes.
3. **Centralité** : calculez au moins deux mesures de centralité et identifiez les 5 nœuds les plus centraux selon chacune. Ces classements convergent-ils ou divergent-ils ?
4. **Communautés** : détectez des communautés avec l'algorithme de Louvain et formulez une hypothèse d'interprétation pour au moins l'une d'entre elles.
5. **Visualisation** : produisez une visualisation finale (statique ou interactive) claire, avec un titre et une légende expliquant les choix de filtrage effectués.

Concluez par quelques lignes de synthèse : qu'apprenez-vous sur la structure relationnelle de votre terrain d'étude, que la seule lecture du tableau de données n'aurait pas permis de voir ?


In [ ]:
# 🧪 Votre code ici — utilisez autant de cellules que nécessaire





<details>
<summary>▶️ Voir un exemple de démarche (terrain A, filtré sur le champ "Law")</summary>

```python
# 1. Construction
links_law = auc_simple[auc_simple["Field"] == "Law"][["Name", "University", "Employer"]].drop_duplicates().dropna()
edges_law = links_law.groupby(["University", "Employer"]).size().reset_index(name="weight")

B_law = nx.Graph()
B_law.add_nodes_from(edges_law["University"].unique(), bipartite=0, node_type="University")
B_law.add_nodes_from(edges_law["Employer"].unique(), bipartite=1, node_type="Employer")
for _, row in edges_law.iterrows():
    B_law.add_edge(row["University"], row["Employer"], weight=row["weight"])

# 2. Description
print(B_law.number_of_nodes(), B_law.number_of_edges())
print(nx.density(B_law))
print(nx.number_connected_components(B_law))

# 3. Centralité (sur la projection université)
univ_nodes_law = {n for n, d in B_law.nodes(data=True) if d["node_type"] == "University"}
G_univ_law = bipartite.weighted_projected_graph(B_law, univ_nodes_law)
pd.DataFrame({
    "degree": nx.degree_centrality(G_univ_law),
    "betweenness": nx.betweenness_centrality(G_univ_law, weight="weight")
}).sort_values("degree", ascending=False).head(5)

# 4. Communautés
communities_law = nx.community.louvain_communities(G_univ_law, weight="weight", seed=42)

# 5. Visualisation
# ... reprendre le code de visualisation de la section 5, appliqué à G_univ_law
```
</details>


# Aide-mémoire

## Glossaire de l'analyse de réseau

| Terme | Description |
|---|---|
| **Nœud (node/vertex)** | Une entité du réseau. |
| **Lien (edge)** | Une relation entre deux nœuds. |
| **Graphe orienté / non-orienté** | Selon que les liens ont ou non un sens (A→B ≠ B→A, ou A–B = B–A). |
| **Graphe pondéré (weighted)** | Chaque lien porte un poids numérique (intensité de la relation). |
| **Réseau biparti (bipartite)** | Deux types de nœuds distincts, les liens n'existant qu'entre les deux types. |
| **Projection** | Transformation d'un réseau biparti en réseau à un seul type de nœuds. |
| **Degré (degree)** | Nombre de liens directs d'un nœud. |
| **Centralité (centrality)** | Famille de mesures de l'importance relative d'un nœud (degré, intermédiarité, proximité, vecteur propre). |
| **Intermédiarité (betweenness)** | Fréquence à laquelle un nœud sert de "pont" entre d'autres nœuds. |
| **Composante connexe (connected component)** | Sous-ensemble de nœuds tous reliés entre eux, directement ou indirectement. |
| **Densité (density)** | Proportion de liens existants par rapport à tous les liens possibles. |
| **Communauté (community)** | Sous-groupe de nœuds densément connectés entre eux. |
| **Modularité (modularity)** | Mesure de la qualité d'un découpage en communautés. |
| **Layout (disposition)** | Algorithme de placement des nœuds dans l'espace pour la visualisation. |
| **Réseau de co-occurrence** | Réseau construit en reliant des entités qui apparaissent ensemble dans un même contexte (document, phrase...). |

## Index des fonctions

| Fonction | Bibliothèque | Rôle |
|---|---|---|
| `nx.Graph()` / `nx.DiGraph()` | networkx | Crée un graphe non-orienté / orienté |
| `G.add_node()` / `G.add_edge()` | networkx | Ajoute un nœud / un lien |
| `G.add_nodes_from()` / `G.add_edges_from()` | networkx | Ajoute plusieurs nœuds/liens depuis une liste |
| `nx.set_node_attributes()` | networkx | Attribue un attribut à plusieurs nœuds depuis un dictionnaire |
| `nx.draw()` / `nx.draw_networkx_nodes/edges/labels()` | networkx | Dessine un graphe (matplotlib) |
| `nx.spring_layout()` | networkx | Calcule une disposition automatique des nœuds |
| `nx.density()` | networkx | Calcule la densité du réseau |
| `nx.number_connected_components()` / `nx.connected_components()` | networkx | Compte / liste les composantes connexes |
| `bipartite.weighted_projected_graph()` | networkx.algorithms.bipartite | Projette un réseau biparti sur un seul type de nœuds |
| `nx.degree_centrality()` | networkx | Centralité de degré |
| `nx.betweenness_centrality()` | networkx | Centralité d'intermédiarité |
| `nx.closeness_centrality()` | networkx | Centralité de proximité |
| `nx.eigenvector_centrality()` | networkx | Centralité de vecteur propre |
| `nx.community.louvain_communities()` | networkx | Détection de communautés (algorithme de Louvain) |
| `nx.community.modularity()` | networkx | Calcule la modularité d'un découpage en communautés |
| `Network()` (pyvis) / `.from_nx()` / `.show()` | pyvis | Crée et exporte une visualisation de réseau interactive (HTML) |
| `itertools.combinations()` | itertools (natif) | Génère toutes les paires possibles d'une liste |
| `collections.Counter()` | collections (natif) | Compte des occurrences efficacement |

## Où trouver de l'aide ?

1. **Documentation officielle** : [NetworkX](https://networkx.org/documentation/stable/), [pyvis](https://pyvis.readthedocs.io/).
2. **The Programming Historian** : plusieurs tutoriels dédiés à l'analyse de réseau en SHS, notamment sur la construction de réseaux à partir de correspondances ou de corpus textuels ([programminghistorian.org](https://programminghistorian.org/)).
3. **Ouvrages de référence** : *Historical Network Research* (Düring, Stark, Keyes, dir.) ; Kadushin, *Understanding Social Networks* — pour les fondements théoriques et méthodologiques, indépendants de tout langage de programmation.
4. **Gephi** ([gephi.org](https://gephi.org/)) : un logiciel libre, à interface graphique, pour l'exploration et la visualisation de réseaux — un bon complément à Python pour l'exploration visuelle interactive, sans code. On peut exporter un graphe NetworkX au format GEXF (`nx.write_gexf(G, "reseau.gexf")`) pour l'ouvrir directement dans Gephi.
5. **Stack Overflow, ChatGPT, Claude** : comme pour tout code Python — donnez le maximum de contexte (message d'erreur complet, structure de vos données, bibliothèque utilisée) pour obtenir une réponse pertinente.

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Pour prolonger cette séance</b> : les trois séances suivent une même logique — partir d'un tableau de données (Séance 1), l'enrichir par du texte (Séance 2), puis en révéler la structure relationnelle (Séance 3). Cette progression est directement transposable à vos propres corpus : toute base de données comportant des colonnes reliant des entités entre elles (auteurs–institutions, correspondants, personnages–scènes, mots-clés co-attribués...) peut devenir un réseau à explorer avec les méthodes vues aujourd'hui.
</div>
